In [1]:
import pandas as pd
import random

In [31]:
wildchat = pd.read_csv("./wildchat_labeled.csv")

/tmp/ipykernel_6025/3041472438.py:1: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  wildchat = pd.read_csv("./wildchat_labeled.csv")


In [27]:
filtered = wildchat[wildchat["init_label"].isin({"beginner", "intermediate", "expert"})][
    ["init_message", "init_label"]
].copy()

# replace newlines with spaces
filtered["init_message"] = filtered["init_message"].str.replace(r'\s+', ' ', regex=True).str.strip()

filtered.to_csv("labeled_wildchat_expertise.csv", index=False)


In [4]:
wildchat["init_message"][9]

'Could you write me an android application that has a login page and can connect to a server'

In [5]:
from openai import OpenAI
SECRET = "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"


SYSTEM_PROMPT = """You are an expert classifier.

Your task is to analyze a single programming-related user message — including both its text and any code — and classify the user's skill level as one of: Beginner, Intermediate, or Expert.

Definitions:
- Beginner: basic programming concepts, syntax, or simple how-to questions.
- Intermediate: practical, applied development, debugging, or framework-level usage.
- Expert: advanced proficiency: model/algorithm design, system architecture, optimization, nontrivial numerical/representation design, or research-level implementation.

Rules:
1. Consider both the text and any code shown.
2. Treat evidence of advanced technical thinking (model internals, numerical/representation design, optimization, architecture, experimental setups) as a strong signal for Expert.
3. Output exactly one word: Beginner, Intermediate, or Expert. No punctuation.
4. When unsure between Intermediate and Expert, choose Expert.
5. NO words OTHER THAN beginner, intermediate, or expert WHATSOEVER.
"""
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=SECRET,
)

In [ ]:
def classify_message(user_message, model="google/gemma-3-27b-it"):  # replace with a deployed OpenAI model you can call
    # We use temperature=0 and restrict max_tokens to keep output deterministic and short.
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message}
        ],
        temperature=0.0,
        top_p=0.3,
        max_tokens=10
        
    )
    # Extract and sanitize output
    label = resp.choices[0].message.content.strip().split()[0]  # first token
    # Normalise label
    label = label.lower()
    if label not in ("beginner","intermediate","expert"):
        # fallback: treat as Intermediate (or you could re-run with more tokens)
        label = None
    return label

In [7]:
from collections import Counter
from tqdm import tqdm
import time
import math

In [23]:
TARGET_PER_CLASS = 1000
OUTPUT_PATH = "./wildchat_labeled.csv"

In [9]:
classify_message("How do i print in python")

'beginner'

In [32]:
beginner_count = sum(wildchat["init_label"] == "beginner")
intermediate_count = sum(wildchat["init_label"] == "intermediate")
expert_count = sum(wildchat["init_label"] == "expert")
rejected = sum(wildchat["init_label"] == "can't classify")

In [33]:
print(f"Starting counts — Beginner: {beginner_count}, Intermediate: {intermediate_count}, Expert: {expert_count}, rejected: {rejected}")

Starting counts — Beginner: 2832, Intermediate: 7226, Expert: 722, rejected: 372


In [24]:

for i in tqdm(range(len(wildchat))):
    msg = wildchat["init_message"][i]
    if msg is None:
        continue
    if isinstance(msg, float) and math.isnan(msg):
        continue
    # Skip if already labeled
    if pd.notna(wildchat.at[i, "init_label"]):
        # print(".", end="")
        continue

    # Stop if all classes hit 300
    if all(x >= TARGET_PER_CLASS for x in [beginner_count, intermediate_count, expert_count]):
        print("✅ Target reached for all classes.")
        break

    # Get label
    label = classify_message(msg)
    if label is None:
        wildchat.at[i, "init_label"] = "can't classify"
        continue
        

    # Skip adding more if that class is full
    # if label == "beginner" and beginner_count >= TARGET_PER_CLASS:
    #     continue
    # if label == "intermediate" and intermediate_count >= TARGET_PER_CLASS:
    #     continue
    # if label == "expert" and expert_count >= TARGET_PER_CLASS:
    #     continue

    # Save label
    wildchat.at[i, "init_label"] = label

    # Update manual counts
    if label == "beginner":
        beginner_count += 1
    elif label == "intermediate":
        intermediate_count += 1
    elif label == "expert":
        expert_count += 1

    # Optional progress printing
    if (i + 1) % 20 == 0:
        print(f"Progress — B:{beginner_count}, I:{intermediate_count}, E:{expert_count}")

    # Small delay to avoid rate limits
    # time.sleep(0.1)

 17%|█▋        | 8353/48391 [00:01<00:05, 7039.63it/s]

Progress — B:2140, I:5387, E:501


 17%|█▋        | 8377/48391 [00:18<02:06, 317.21it/s] 

Progress — B:2143, I:5400, E:504
Progress — B:2149, I:5413, E:505


 17%|█▋        | 8418/48391 [00:37<06:18, 105.53it/s]

...

Progress — B:2817, I:7205, E:721


 23%|██▎       | 11180/48391 [27:02<4:02:18,  2.56it/s]

Progress — B:2825, I:7216, E:722


 23%|██▎       | 11199/48391 [27:14<1:30:29,  6.85it/s]


KeyboardInterrupt: 

In [25]:
wildchat.to_csv(OUTPUT_PATH, index=False)